In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')
#os.chdir("/content/drive/MyDrive")

Mounted at /content/drive


In [5]:
#Load the CSV File

df = pd.read_csv("/content/drive/MyDrive/VAT datasets/ACCdata/accelerometer_2026-03-14_walking.csv")

print(df.head())

       time     gFx     gFy     gFz   TgF
0  0.041973 -0.2267  0.2121  0.8727  0.93
1  0.070976 -0.0362  0.3381  0.9196  0.98
2  0.094008 -0.0186  0.4290  0.8003  0.91
3  0.116441 -0.0645  0.4456  0.7290  0.86
4  0.139456 -0.1046  0.4339  0.7398  0.86


In [6]:
#Remove Metadata Rows (if present)

#Sometimes the file contains header information before the data.

df = df[['time','gFx','gFy','gFz','TgF']]
df = df.dropna()

In [7]:
##Create Time Windows

#Human activity recognition usually uses 2–5 second windows.

#Example: 2-second window

window_size = 400   # if sampling rate ≈ 200 Hz

In [8]:
#Feature Extraction Function
def extract_features(window):

    features = {}

    features['mean_x'] = window['gFx'].mean()
    features['mean_y'] = window['gFy'].mean()
    features['mean_z'] = window['gFz'].mean()

    features['std_x'] = window['gFx'].std()
    features['std_y'] = window['gFy'].std()
    features['std_z'] = window['gFz'].std()

    features['max_x'] = window['gFx'].max()
    features['max_y'] = window['gFy'].max()
    features['max_z'] = window['gFz'].max()

    features['min_x'] = window['gFx'].min()
    features['min_y'] = window['gFy'].min()
    features['min_z'] = window['gFz'].min()

    features['mean_magnitude'] = window['TgF'].mean()
    features['std_magnitude'] = window['TgF'].std()

    return features

In [9]:
#Apply Feature Extraction to Windows
feature_list = []

for start in range(0, len(df), window_size):

    window = df.iloc[start:start+window_size]

    if len(window) == window_size:
        feats = extract_features(window)
        feature_list.append(feats)

features_df = pd.DataFrame(feature_list)

In [11]:
features_df.head()

,mean_x,mean_y,mean_z,std_x,std_y,std_z,max_x,max_y,max_z,min_x,min_y,min_z,mean_magnitude,std_magnitude
0,0.005560,0.408954,0.895612,0.093915,0.111409,0.123259,0.2511,0.6225,1.3974,-0.2267,0.1300,0.6010,0.995800,0.119780
1,0.011444,0.422417,0.885783,0.094347,0.104916,0.141410,0.2883,0.6069,1.3427,-0.2130,0.1612,0.5013,0.992125,0.136353
2,-0.006186,0.455325,0.868505,0.104366,0.106634,0.125376,0.2756,0.6880,1.2039,-0.2492,0.2101,0.5609,0.992750,0.118579
3,-0.010524,0.489387,0.842858,0.101217,0.117435,0.116456,0.2707,0.7388,1.1824,-0.2199,0.1808,0.5834,0.987275,0.112151


In [ ]:
#Each row now represents one time window.

Add Activity Labels (Important for ML)

If collecting activity data like:

walking

sitting

running

Add a label column:

In [12]:
features_df['activity'] = 'walking'

In [13]:
features_df.to_csv("ml_featuresHARwalking.csv", index=False)